# 10–11｜轨迹级失败分析与评估完整性

对应[第 10 章](../course/10-failure-analysis-and-robustness.md)和[第 11 章](../course/11-evaluation-integrity.md)。本 Notebook 用真实 compact evidence 与精选轨迹练习“聚合 → 阶段 → 机制 → 协议”的诊断链。

## 学习目标

验证 failure classes 互斥且完备；用事件时间线区分 reached/no-lift 与成功；根据分母而非失败数量判断弱区；识别 guide-state 初始状态泄漏；只提出证据支持的下一项干预。

## 本节知识地图

本节只学 6 个诊断概念。分析顺序不能颠倒：先证据完整性，再失败机制。

| 知识点 | 一句话解释 | 项目中的例子 | 掌握检查 |
| --- | --- | --- | --- |
| 分类守恒 | 失败类别互斥且总数等于全部失败 | 60 个失败 | 重算类别和 |
| 轨迹里程碑 | 用事件首次发生时间定位行为链断点 | reached/lift/success | 画成功与失败时间线 |
| 条件成功率 | 判断弱区必须同时有 successes 和 episodes | y-position bins | 不只数失败点 |
| 探索与确认 | 发现弱区的数据不能同时冒充独立确认 | original vs fixed-left | 指出协议差异 |
| guide-state 泄漏 | 训练辅助进入评估会虚增标准 reset 表现 | 5% `picked` reset | 识别 step-1 reached |
| 机制干预 | 改动针对 dominant failure，并有可否定预测 | stable post-contact lift | 写唯一干预 |

## 关键概念与符号

| 名词 | 含义 | 不要混淆 |
| --- | --- | --- |
| failure count | 某类失败的数量 | 没有分母时不是失败率 |
| curated trajectory | 为理解字段选择的个例 | 不是随机 rate sample |
| effective config | 运行中真正生效并写入报告的配置 | 不只看默认配置 |
| historical evidence | 旧协议下仍保留的开发证据 | 不能与新正式结果合并 |
| guide-free | reset 中强制 guide probability=0 | 不能靠删旧轨迹伪造 |

> **诊断顺序：** 协议有效 → 分类守恒 → 条件率有分母 → 时间线找瓶颈 → 单变量确认。

## 先预测

1. 60 个失败中 55 个 `reached_no_lift`，下一步应优先改 approach、contact 还是 post-contact lift？
2. 成功和失败都 100% 曾双指接触，通用 contact bonus 的可证伪预测是什么？
3. 历史评估恰有约 5% 回合在 step 1 reached，最先检查哪项 reset 配置？
4. 能否删除这 5% 记录后把剩余样本称为 guide-free 正式评估？

## 运行与观察

加载 guide-free 正式 compact report、历史 guide-assisted report、位置分层结果和 8 条精选 episode。

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(ROOT / 'docs' / 'notebooks'))

import matplotlib.pyplot as plt
import numpy as np
from course_utils import assert_course_kernel, load_json, trajectory_events

assert_course_kernel(ROOT)
formal = load_json(ROOT / 'reproduction/results/linux-guide-free-left-trajectory-analysis.json')
historical = load_json(ROOT / 'reproduction/results/linux-trajectory-failure-analysis.json')
position = load_json(ROOT / 'reproduction/results/linux-position-stratified-analysis.json')
fixture = load_json(ROOT / 'docs/data/guide-free-left-episodes-fixture.json')
examples = fixture['curated_episode_examples']
print('Loaded formal episodes:', formal['protocol']['episodes'])
print('Loaded curated examples:', len(examples), '(teaching only)')

### 1. 先检查分类守恒，再看 dominant class

互斥分类必须满足 `sum(class counts) == failures`。否则 dominant class 的比例没有可信分母。

In [ ]:
counts = formal['failure_classification']['counts']
failures = formal['aggregate']['failures']
assert sum(counts.values()) == failures
dominant = max(counts, key=counts.get)
dominant_share = counts[dominant] / failures
print(f'dominant = {dominant}: {counts[dominant]}/{failures} = {dominant_share:.2%}')

labels = list(counts)
values = [counts[label] for label in labels]
colors = ['tab:orange' if label == dominant else 'tab:blue' for label in labels]
plt.figure(figsize=(9, 4))
plt.barh(labels, values, color=colors)
plt.xlabel('failed episodes'); plt.title('Guide-free mutually exclusive failure classes')
plt.tight_layout(); plt.show()

### 2. 成功与 reached-no-lift 的事件时间线

精选记录不能估计 rate，但适合学习字段语义。成功轨迹经过 reached → lifted → success；dominant failure reached 后直到 200 步都没有 lift。

In [ ]:
success_example = next(row for row in examples if row['success'])
failure_example = next(row for row in examples if row['failure_class'] == 'reached_no_lift')

fig, ax = plt.subplots(figsize=(10, 3.5))
for y, record, color in [(1, success_example, 'tab:green'), (0, failure_example, 'tab:red')]:
    ax.hlines(y, 0, record['episode_length'], color=color, linewidth=3, alpha=0.35)
    for label, step in trajectory_events(record):
        ax.scatter(step, y, s=70, color=color)
        ax.text(step, y + 0.12, f'{label} @{step}', ha='center', fontsize=9)
ax.set(yticks=[0, 1], yticklabels=[failure_example['episode_id'], success_example['episode_id']], xlabel='control step', title='Curated trajectory milestones (not a rate sample)')
ax.set_ylim(-0.45, 1.5); ax.grid(axis='x', alpha=0.25); plt.show()
print('success max z:', success_example['max_box_height'])
print('failure max z:', failure_example['max_box_height'])

### 3. 接触不是当前区分信号

若成功和失败都已获得双指接触，再奖励“发生过接触”对两类轨迹缺少区分力。更具体的假设应针对接触后的稳定抬升，而不是凭直觉重复加强 approach/contact。

In [ ]:
comparison = formal['grasp_acquisition_comparison']
metrics = {
    'success bilateral contact': comparison['success']['ever_bilateral_contact_rate'],
    'failure bilateral contact': comparison['failure']['ever_bilateral_contact_rate'],
    'failure lifted': comparison['failure']['lifted_rate'],
}
for name, value in metrics.items():
    print(f'{name:28s} {value:.2%}')
assert metrics['success bilateral contact'] == 1.0
assert metrics['failure bilateral contact'] == 1.0
assert metrics['failure lifted'] < 0.1

### 4. 位置弱区必须同时有分母

失败集中在负 y 是线索；条件成功率才是比较。下面比较探索性分层，不能在看图后继续缩 bin 并把同一数据当确认实验。

In [ ]:
groups = position['grouped_results']
chosen_groups = ['y_below_negative_0_02', 'y_at_or_above_negative_0_02', 'hard_bin_negative_0_03_to_negative_0_02']
group_rates = [groups[name]['success_rate'] for name in chosen_groups]
group_ns = [groups[name]['episodes'] for name in chosen_groups]
for name, n, group_rate in zip(chosen_groups, group_ns, group_rates):
    print(f'{name:44s} {groups[name]["successes"]}/{n} = {group_rate:.2%}')
plt.figure(figsize=(9, 3.5))
plt.bar(['y < -0.02', 'y ≥ -0.02', 'hard bin'], group_rates, color=['tab:orange', 'tab:blue', 'tab:red'])
plt.axhline(0.90, color='0.25', linestyle='--', label='90% reference')
plt.ylim(0.8, 1.0); plt.ylabel('conditional success rate'); plt.legend(); plt.show()

### 5. Guide-state 完整性审计

历史 schema-3 复用了训练的 5% guide 起始状态；50 个 step-1 reached 且全部成功是强线索。正式 schema-4 强制 guide probability=0。协议不同，不能合并、相减或筛掉 50 条后冒充重采集。

In [ ]:
old_protocol = historical['protocol']
integrity = historical['evaluation_integrity_finding']
new_protocol = formal['protocol']
rows = [
    ('status', old_protocol['status'], new_protocol['status']),
    ('schema', old_protocol['report_schema_version'], new_protocol['report_schema_version']),
    ('guide probability', integrity['configured_guide_swap_probability'], new_protocol['guide_swap_probability']),
    ('step-1 reached', integrity['step_1_reached_episodes'], 'excluded by protocol'),
]
print(f'{"field":20s} {"historical":28s} formal')
print('-' * 75)
for field, old, new in rows:
    print(f'{field:20s} {str(old):28s} {new}')

## 动手修改

把 `selected_episode_id` 换成 fixture 中其他成功或失败 ID，先预测事件序列与最大高度，再运行。最后恢复 `101:0004`。

In [ ]:
selected_episode_id = '101:0004'
selected = next(row for row in examples if row['episode_id'] == selected_episode_id)
print('episode:', selected['episode_id'])
print('success:', selected['success'], 'failure class:', selected['failure_class'])
print('events:', trajectory_events(selected))
print('max box height:', selected['max_box_height'])

## 自测

以下断言把守恒、机制证据、精选样例边界和协议修复串起来。运行前逐条说出若失败意味着什么。

In [ ]:
assert dominant == 'reached_no_lift' and counts[dominant] == 55
assert dominant_share > 0.9
assert fixture['fixture_role'].endswith('not_a_rate_sample')
assert selected_episode_id == '101:0004' and selected['failure_class'] == dominant
assert old_protocol['status'] == 'historical_guide_assisted'
assert integrity['configured_guide_swap_probability'] == 0.05
assert integrity['step_1_reached_episodes'] == 50
assert new_protocol['guide_swap_probability'] == 0.0
assert new_protocol['report_schema_version'] > old_protocol['report_schema_version']
print('PASS: exhaustive classes, trajectory mechanism, position denominator, and guide-free protocol')

## 学完请记住

关闭本页后，你应能脱稿说出：

1. 失败分类先满足互斥、完备和计数守恒，dominant class 才可信；
2. 弱区需要条件成功率，失败散点本身没有分母；
3. 精选轨迹适合解释阶段，不适合估计总体 rate；
4. 本项目 dominant failure 是 `reached_no_lift`，不是未接触；
5. guide-state 泄漏必须修复协议并重采，不能删旧样本补救；
6. 下一项干预应针对接触后的稳定抬升，并设置等计算量 control。

若无法提出下一步，先定位自己缺的是协议、分母、分类还是时间线，不要直接猜 reward。

## 反思与记录

在 `notes/10-11-notebook-reflection.md` 写出完整链：异常 → 源码配置 → 协议修复 → schema 测试 → 重新采集 → 历史结论降级。然后填写一个机制假设：

- dominant failure：
- 唯一干预：
- 中间指标：
- 会否定假设的结果：
- 为什么必须有等 3M 步 control：

Notebook 只到 Gate 4 PRACTICED；自己的全量 guide-free report 才能到 READY。